# C1 CNN from scratch: runs on a Colab T4

Kernel: **Colab, T4 GPU** (Runtime → Change runtime type → T4). Run all cells top to bottom.

It runs a 1-minute GPU smoke test, then **every tuning stage written in `c1/cnn/configs.py`** (seeds 0–4, with the reps per stage from `STAGE_REPS`), the training curves, the per-stage summaries,the `best`/`fast` report-seed runs once they're chosen, and the figures.

All rows are tagged `hw=colab-t4`, so every time in the report comes from the same GPU type as the rest of the team. Laptop rows (`hw=local-cpu`) are never committed and never reach Colab.

**Before committing this notebook, clear its outputs** (Edit → Clear All Outputs); outputs are saved into the file and the repo is public.

**Resumable:** rows go to a copy of `results/ryan.csv` on your Google Drive, so they survive a disconnect or runtime reset. Run all cells again and rows already logged are skipped.

**Nothing is pushed from Colab.** The notebook clones the repo read-only (the repo is public, so no token is needed); `BRANCH` in cell 1 picks the branch, and it has to be on GitHub with the latest code. Results stay in the Drive folder below: download it to bring them back.

If you re-run after a change, **restart the kernel first**, because `common.data` only reads `COMP433_DATA` once, at import.

In [ ]:
# 1. Get the code onto the Colab machine (read-only clone of the branch)
import os
import shutil

REPO = "/content/repo"
BRANCH = "main"   # the branch to run; set it to a feature branch to test unmerged code
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/MassPaol/comp433-final-project.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip install -q -e . -r requirements.txt

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Copy CIFAR-10 from the team folder to the Colab machine (faster than reading from Drive)
DRIVE_TAR = "/content/drive/MyDrive/comp433_cache/data/cifar-10-python.tar.gz"  # comp433_cache shortcut in My Drive
LOCAL_DATA = "/content/data"

assert os.path.exists(DRIVE_TAR), (
    f"{DRIVE_TAR} not found: add the team's comp433_cache folder as a shortcut in My Drive")
if not os.path.exists(f"{LOCAL_DATA}/cifar-10-python.tar.gz"):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copy(DRIVE_TAR, LOCAL_DATA)  # torchvision unpacks it on first load

os.environ["COMP433_DATA"] = LOCAL_DATA

In [ ]:
# 4. Import the project code (must come after COMP433_DATA is set) and check the data loads
import torch
from common import data

assert data.DATA_ROOT == os.environ["COMP433_DATA"], (
    "common.data was imported before COMP433_DATA was set. Restart the kernel and run all cells again.")
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (running on CPU)")

X, y, Xv, yv, classes = data.load_instance(0)
print(X.shape, Xv.shape, [data.CLASS_NAMES[c] for c in classes])  # expect (100, 32, 32, 3) (700, 32, 32, 3)

In [ ]:
# 5. Make sure this really is a T4 (every row gets tagged hw=colab-t4), and log to Drive
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(), (
    f"Expected a T4 GPU, got {torch.cuda.get_device_name() if torch.cuda.is_available() else 'no GPU'}")
HW = "colab-t4"

RESULTS_DIR = "/content/drive/MyDrive/comp433_c1_cnn_results"
RESULTS = f"{RESULTS_DIR}/ryan.csv"
CURVES = f"{RESULTS_DIR}/ryan_curves.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(RESULTS):
    shutil.copy("results/ryan.csv", RESULTS)  # start from the repo's rows
print("logging to", RESULTS)

In [ ]:
# 6. GPU smoke test (~1 min): 2-epoch runs of the first stage on 2 seeds, logged to a throwaway folder.
#    Catches CUDA / AMP problems before the long runs. Nothing here goes into the results.
from c1.cnn import configs
SMOKE_DIR = "/content/smoke"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
first = next(iter(configs.STAGES))
!python -W ignore -m c1.cnn.run --stage {first} --seeds tune --reps 1 --hw smoke --results-dir {SMOKE_DIR} --smoke-epochs 2
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

In [ ]:
# 7. Run every tuning stage written in configs.py, in order, on seeds 0-4.
#    Reps per stage come from configs.STAGE_REPS. Resumable: finished (config, seed, rep) rows are skipped,
#    so after a disconnect just run all cells again. Limit with e.g. STAGES_TO_RUN = ["s2a"].
STAGES_TO_RUN = list(configs.STAGES)
print("stages:", {s: f"{len(configs.STAGES[s])} configs x {configs.STAGE_REPS[s]} reps" for s in STAGES_TO_RUN})
for stage in STAGES_TO_RUN:
    print(f"\n===== {stage} =====")
    !python -W ignore -m c1.cnn.run --stage {stage} --seeds tune --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 8. Training curves (tuning seeds only) for configs.CURVE_CONFIGS, for the dynamics figure. Resumable.
!python -W ignore -m c1.cnn.curves --every 10 --hw {HW} --out {CURVES}

In [ ]:
# 9. Per-stage summary + selection rule on the T4 rows
for stage in STAGES_TO_RUN:
    print(f"\n===== {stage} (T4) =====")
    !python -m c1.cnn.select --hw {HW} --results {RESULTS} --stage {stage}
# Paired comparison of two configs, e.g.:
# !python -m c1.cnn.select --hw {HW} --results {RESULTS} --pair s0_flipcrop s0_noaug

In [ ]:
# 10. Final: best and fast on the report seeds 10-19 (only once FINAL_SOURCE is filled in configs.py)
if configs.FINAL_SOURCE:
    !python -W ignore -m c1.cnn.run --config best fast --seeds report --hw {HW} --results-dir {RESULTS_DIR}
    !python -m c1.cnn.select --hw {HW} --results {RESULTS} --seeds report --pair best fast
else:
    print("FINAL_SOURCE is empty: still tuning, nothing to run on the report seeds.")

In [ ]:
# 11. Draw the report figures from the T4 rows into the Drive folder (download them with the CSVs)
FIG_DIR = f"{RESULTS_DIR}/figures"
!python -m c1.cnn.figures --hw {HW} --results {RESULTS} --curves {CURVES} --outdir {FIG_DIR}